# Drawdown-aware Reward Proxy v1

Runtime card: `runtime/reward/card/drawdown_aware_reward_proxy_v1`

This notebook continues from `drawdown_aware_objective_screen_v1`. The previous screen proved that strict full-window selection can find `+30pp` / `<15%` candidates, but every strict-pass candidate is blocked by saved walk-forward risk or rebound underparticipation.

This run tests whether candidate reward terms would select a less overfit path before opening a PPO branch. It uses existing daily paths as offline trajectories and screens reward-proxy combinations: log return, current path drawdown, expected/future drawdown proxy, rebound opportunity cost, bear exposure, and turnover.

## Governance

- **Direction**: `reward`.
- **Role**: `diagnose`.
- **Changed layer**: `reward`; no shared `src` changes in this run.
- **Canonical baseline**: current qlib SOXL harness and retained model families.
- **Direct comparison baseline**: `drawdown_aware_objective_screen_v1` and the strict-only selector candidates it exposed.
- **Target windows**: strict OOS plus available saved walk-forward daily paths (`wf_2020`, `wf_2022`, `wf_2023_2024`).
- **Hypothesis**: a scalar reward proxy with drawdown and rebound-opportunity terms should demote strict-only overfit candidates; if it cannot, the next real training branch needs regime-conditioned reward/architecture rather than more scalar weight tuning.
- **Acceptance criteria**: identify at least one reward proxy that selects a candidate with better walk-forward risk/excess trade-off, or prove the current path pool has no scalar proxy satisfying the target.
- **Failure mode**: the proxy either selects high-return/high-DD recovery paths or low-DD/underparticipating paths, proving the need for state-conditioned objectives.
- **Comparison-table row key**: `drawdown_aware_reward_proxy_v1`.


## External Basis

Primary sources used for this reward proxy design:

- Qlib keeps the loop inside an AI quant research platform with explicit infrastructure for data-driven workflows: https://arxiv.org/abs/2009.11189 and https://github.com/microsoft/qlib
- DeepTrader embeds market conditions and uses negative maximum drawdown as the reward function for risk-return balance: https://ojs.aaai.org/index.php/AAAI/article/view/16144
- Expected Maximum Drawdown RRL uses expected max drawdown / Calmar-style objectives for trading policy optimization: https://doi.org/10.1016/j.eswa.2017.06.023
- FinRL/FinRL-X separates market environments, DRL agents, financial applications, and risk controls; this supports keeping reward research separate from broker/runtime code: https://github.com/AI4Finance-Foundation/FinRL
- Risk-aware reward design literature supports multi-term rewards that combine returns and risk-adjusted components, but this notebook treats weight tuning as diagnostic only: https://arxiv.org/abs/2506.04358


In [1]:
from __future__ import annotations

import itertools
import json
import math
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists():
            return path
    raise RuntimeError(f'Could not find repo root from {start}')


ROOT = find_repo_root(Path.cwd())
CARD_DIR = ROOT / 'runtime/reward/card/drawdown_aware_reward_proxy_v1'
CARD_DIR.mkdir(parents=True, exist_ok=True)

STRICT_RELATIVE_TARGET = 30.0
STRICT_MAX_DD_TARGET = 15.0
WF_RELATIVE_FLOORS = {'wf_2020': -10.0, 'wf_2022': 30.0, 'wf_2023_2024': 30.0}

pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 180)
print(f'card_dir={CARD_DIR.relative_to(ROOT)}')


card_dir=runtime/reward/card/drawdown_aware_reward_proxy_v1


## Daily Path Loader

The loader scans existing `*_soxl_daily.csv` runtime artifacts. For each trajectory it computes:

- return, buy-hold relative return, MaxDD, avg position, turnover.
- path drawdown penalty: average squared drawdown beyond 15%.
- expected/future drawdown proxy: forward 21-day equity drawdown beyond 8%.
- rebound opportunity cost: underexposure during high-volatility positive-momentum rebound states.
- bear exposure penalty: exposure during negative momentum / drawdown states.

The proxy uses lagged price-derived states where possible. It is a diagnostic for reward design, not a claim that all terms are directly usable without implementation review.


In [2]:
def infer_family(path: Path) -> str:
    parts = path.parts
    if 'card' in parts:
        idx = parts.index('card')
        if idx + 1 < len(parts):
            return parts[idx + 1]
    return path.parent.name


def infer_window(path: Path) -> str:
    text = str(path)
    for wf in WF_RELATIVE_FLOORS:
        if f'/{wf}/' in text or f'_{wf}_' in path.name:
            return wf
    return 'strict_oos'


def _safe_pct(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors='coerce').pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)


def max_drawdown(nav: pd.Series) -> float:
    nav = pd.to_numeric(nav, errors='coerce').dropna()
    if len(nav) < 2:
        return np.nan
    dd = nav / nav.cummax() - 1.0
    return float(-dd.min() * 100.0)


def forward_drawdown(nav: pd.Series, horizon: int = 21) -> pd.Series:
    nav = pd.to_numeric(nav, errors='coerce').astype(float)
    values = nav.to_numpy()
    out = np.zeros(len(values), dtype=float)
    for i in range(len(values)):
        end = min(len(values), i + horizon + 1)
        if i >= end or not np.isfinite(values[i]) or values[i] <= 0:
            out[i] = 0.0
            continue
        future_min = np.nanmin(values[i:end])
        out[i] = max(0.0, 1.0 - future_min / values[i])
    return pd.Series(out, index=nav.index)


def path_metrics(path: Path) -> dict[str, Any] | None:
    try:
        df = pd.read_csv(path)
    except Exception:
        return None
    required = {'strategy_nav', 'price', 'position'}
    if not required.issubset(df.columns) or len(df) < 40:
        return None

    df = df.copy()
    df['datetime'] = pd.to_datetime(df.get('datetime'), errors='coerce')
    df = df.dropna(subset=['datetime']).sort_values('datetime').reset_index(drop=True)
    for col in ['strategy_nav', 'buyhold_nav', 'price', 'position']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'buyhold_nav' not in df.columns or df['buyhold_nav'].isna().all():
        first_price = df['price'].dropna().iloc[0]
        df['buyhold_nav'] = df['price'] / first_price * df['strategy_nav'].dropna().iloc[0]

    df = df.dropna(subset=['strategy_nav', 'buyhold_nav', 'price', 'position'])
    if len(df) < 40:
        return None

    strategy_ret = _safe_pct(df['strategy_nav'])
    buyhold_ret = _safe_pct(df['buyhold_nav'])
    price_ret = _safe_pct(df['price'])
    log_return_sum = float(np.log1p(strategy_ret.clip(lower=-0.99)).sum())
    strategy_return = float((df['strategy_nav'].iloc[-1] / df['strategy_nav'].iloc[0] - 1.0) * 100.0)
    buyhold_return = float((df['buyhold_nav'].iloc[-1] / df['buyhold_nav'].iloc[0] - 1.0) * 100.0)
    relative_return = strategy_return - buyhold_return
    dd = df['strategy_nav'] / df['strategy_nav'].cummax() - 1.0
    current_dd_penalty = float(np.square(np.maximum(0.0, -dd - 0.15)).mean())
    future_dd = forward_drawdown(df['strategy_nav'], horizon=21)
    future_dd_penalty = float(np.square(np.maximum(0.0, future_dd - 0.08)).mean())

    pos = df['position'].clip(lower=0.0, upper=1.5)
    pos_delta = pos.diff().abs().fillna(0.0)
    turnover = float(pos_delta.sum())
    turnover_per_day = float(pos_delta.mean())
    trade_count = int((pos_delta > 1e-4).sum())

    price_mom_21 = df['price'].pct_change(21).fillna(0.0)
    price_mom_5 = df['price'].pct_change(5).fillna(0.0)
    price_drawdown_63 = df['price'] / df['price'].rolling(63, min_periods=5).max() - 1.0
    vol_21 = price_ret.rolling(21, min_periods=5).std().fillna(0.0)
    vol_threshold = float(vol_21.quantile(0.70)) if len(vol_21) else 0.0
    rebound_state = (price_mom_21 > 0.20) & (price_mom_5 > 0.02) & (vol_21 >= vol_threshold)
    if {'mom_21', 'relative_strength_21', 'vol_of_vol_21'}.issubset(df.columns):
        feature_rebound = (pd.to_numeric(df['mom_21'], errors='coerce') > 0.45) & (pd.to_numeric(df['relative_strength_21'], errors='coerce') > 1.0)
        rebound_state = rebound_state | feature_rebound.fillna(False)
    rebound_opportunity_cost = float(((1.0 - pos.clip(upper=1.0)) * buyhold_ret.clip(lower=0.0) * rebound_state.astype(float)).sum())
    rebound_days = int(rebound_state.sum())

    bear_state = (price_mom_21 < -0.15) | (price_drawdown_63 < -0.25)
    bear_exposure_penalty = float((pos * (-buyhold_ret).clip(lower=0.0) * bear_state.astype(float)).sum())
    bear_days = int(bear_state.sum())

    return {
        'path': str(path.relative_to(ROOT)),
        'family': infer_family(path),
        'window': infer_window(path),
        'rows': int(len(df)),
        'start': str(df['datetime'].iloc[0].date()),
        'end': str(df['datetime'].iloc[-1].date()),
        'return_pct': strategy_return,
        'buyhold_return': buyhold_return,
        'relative_return': relative_return,
        'max_dd': max_drawdown(df['strategy_nav']),
        'avg_pos': float(pos.mean()),
        'trade_count': trade_count,
        'turnover': turnover,
        'turnover_per_day': turnover_per_day,
        'log_return_sum': log_return_sum,
        'current_dd_penalty': current_dd_penalty,
        'future_dd_penalty': future_dd_penalty,
        'rebound_opportunity_cost': rebound_opportunity_cost,
        'rebound_days': rebound_days,
        'bear_exposure_penalty': bear_exposure_penalty,
        'bear_days': bear_days,
    }


paths = sorted(ROOT.glob('runtime/**/*_soxl_daily.csv'))
metrics = [m for p in paths if (m := path_metrics(p)) is not None]
path_table = pd.DataFrame(metrics)
print(f'daily paths loaded: {len(path_table)} / {len(paths)}')
display(path_table.groupby('window').size().rename('paths').reset_index())
display(path_table.head(5))


daily paths loaded: 88 / 90


,window,paths
0,strict_oos,37
1,wf_2020,17
2,wf_2022,17
3,wf_2023_2024,17


,path,family,window,rows,start,end,return_pct,buyhold_return,relative_return,max_dd,avg_pos,trade_count,turnover,turnover_per_day,log_return_sum,current_dd_penalty,future_dd_penalty,rebound_opportunity_cost,rebound_days,bear_exposure_penalty,bear_days
0,runtime/execution/card/recovery_reentry_2021_t...,recovery_reentry_2021_trendbreak_cap_v1,strict_oos,1605,2020-01-02,2026-05-21,1023.710464,816.512500,207.197964,30.436657,0.408919,209,36.390924,0.022673,2.419221,0.000498,0.000808,2.355095,80,5.674240,625
1,runtime/execution/card/recovery_reentry_2021_t...,recovery_reentry_2021_trendbreak_cap_v1,wf_2020,252,2020-01-02,2020-12-30,83.651063,58.093571,25.557493,26.139320,0.368127,27,6.232652,0.024733,0.607867,0.001350,0.000698,0.225628,13,1.162319,79
2,runtime/execution/card/recovery_reentry_2021_t...,recovery_reentry_2021_trendbreak_cap_v1,wf_2022,250,2022-01-03,2022-12-29,-23.981110,-86.588072,62.606962,25.936720,0.171215,20,2.166701,0.008667,-0.274188,0.002379,0.000580,0.718389,10,1.450868,211
3,runtime/execution/card/recovery_reentry_2021_t...,recovery_reentry_2021_trendbreak_cap_v1,wf_2023_2024,251,2024-01-02,2024-12-30,45.740106,-0.356635,46.096740,28.204933,0.299847,24,4.014522,0.015994,0.376655,0.001652,0.000368,0.051093,2,0.522062,120
4,runtime/execution/card/recovery_reentry_anchor...,recovery_reentry_anchor_confirm_v1,strict_oos,1605,2020-01-02,2026-05-21,1894.611414,816.512500,1078.098915,41.587207,0.466852,202,36.573357,0.022787,2.993034,0.001072,0.002069,3.970062,164,7.273566,625


## Proxy Reward Variants

Scores are z-normalized across strict-OOS paths to avoid unit scale artifacts. A variant selects a strict path using only strict-OOS information; then we evaluate that path's family on available walk-forward paths.

This intentionally mirrors the overfit question: if a reward selects a good-looking strict path but its family fails walk-forward, the reward is not sufficient for promotion.


In [3]:
strict_paths = path_table[(path_table['window'] == 'strict_oos') & (path_table['rows'] >= 200)].copy()
wf_paths = path_table[path_table['window'].isin(WF_RELATIVE_FLOORS)].copy()

FEATURES = [
    'log_return_sum',
    'max_dd',
    'current_dd_penalty',
    'future_dd_penalty',
    'rebound_opportunity_cost',
    'bear_exposure_penalty',
    'turnover_per_day',
]

for col in FEATURES:
    mean = strict_paths[col].mean()
    std = strict_paths[col].std(ddof=0)
    strict_paths[f'z_{col}'] = 0.0 if std == 0 or pd.isna(std) else (strict_paths[col] - mean) / std

# Collapse to one strict candidate per family/path. Some families have multiple strict files; keep all in the screen.
wf_agg = (
    wf_paths.groupby(['family', 'window'])
    .agg(
        wf_return=('return_pct', 'max'),
        wf_relative_return=('relative_return', 'max'),
        wf_max_dd=('max_dd', 'min'),
        wf_avg_pos=('avg_pos', 'mean'),
        wf_paths=('path', 'count'),
    )
    .reset_index()
)

supplemental_wf_rows = []
predictive_selection = ROOT / 'runtime/model_selection/card/predictive_drawdown_risk_gate_v1/selection.json'
if predictive_selection.exists():
    payload = json.loads(predictive_selection.read_text(encoding='utf-8'))
    for row in payload.get('walkforward', []):
        supplemental_wf_rows.append({
            'family': 'predictive_drawdown_risk_gate_v1',
            'window': row['window'],
            'wf_return': float(row.get('return', np.nan)),
            'wf_relative_return': float(row.get('relative_return', np.nan)),
            'wf_max_dd': float(row.get('max_dd', np.nan)),
            'wf_avg_pos': float(row.get('avg_pos', np.nan)),
            'wf_paths': 0,
        })

residual_wf = ROOT / 'runtime/execution/card/risk_budget_residual_drawdown_guard_walkforward_v1/walkforward_comparison.csv'
if residual_wf.exists():
    comp = pd.read_csv(residual_wf)
    for _, row in comp.iterrows():
        supplemental_wf_rows.append({
            'family': 'risk_budget_residual_drawdown_guard_v1',
            'window': row['window'],
            'wf_return': np.nan,
            'wf_relative_return': float(row.get('residual_relative_return', np.nan)),
            'wf_max_dd': float(row.get('residual_max_dd', np.nan)),
            'wf_avg_pos': np.nan,
            'wf_paths': 0,
        })

if supplemental_wf_rows:
    wf_agg = pd.concat([wf_agg, pd.DataFrame(supplemental_wf_rows)], ignore_index=True)
    wf_agg = (
        wf_agg.sort_values(['family', 'window', 'wf_max_dd', 'wf_relative_return'], ascending=[True, True, True, False])
        .drop_duplicates(['family', 'window'], keep='first')
        .reset_index(drop=True)
    )
print(f'supplemental_wf_rows={len(supplemental_wf_rows)}')
wide_wf = wf_agg.pivot(index='family', columns='window')
wide_wf.columns = [f'{metric}_{window}' for metric, window in wide_wf.columns]
wide_wf = wide_wf.reset_index()

screen_base = strict_paths.merge(wide_wf, on='family', how='left')
for wf in WF_RELATIVE_FLOORS:
    screen_base[f'wf_{wf}_dd_gap'] = np.maximum(0.0, screen_base.get(f'wf_max_dd_{wf}', np.nan) - STRICT_MAX_DD_TARGET)
    screen_base[f'wf_{wf}_rel_gap'] = np.maximum(0.0, WF_RELATIVE_FLOORS[wf] - screen_base.get(f'wf_relative_return_{wf}', np.nan))

def outcome_fields(row: pd.Series) -> dict[str, Any]:
    strict_pass = bool(row['relative_return'] >= STRICT_RELATIVE_TARGET and row['max_dd'] <= STRICT_MAX_DD_TARGET)
    wf_missing = 0
    wf_loss = 0.0
    wf_guard_pass = True
    for wf, rel_floor in WF_RELATIVE_FLOORS.items():
        rel = row.get(f'wf_relative_return_{wf}', np.nan)
        dd = row.get(f'wf_max_dd_{wf}', np.nan)
        if pd.isna(rel) or pd.isna(dd):
            wf_missing += 1
            wf_guard_pass = False
            continue
        dd_gap = max(0.0, dd - STRICT_MAX_DD_TARGET)
        rel_gap = max(0.0, rel_floor - rel)
        wf_loss += 2.0 * dd_gap ** 2 + 0.25 * rel_gap ** 2
        if dd > STRICT_MAX_DD_TARGET or rel < rel_floor:
            wf_guard_pass = False
    strict_loss = max(0.0, STRICT_RELATIVE_TARGET - row['relative_return']) ** 2 + 4.0 * max(0.0, row['max_dd'] - STRICT_MAX_DD_TARGET) ** 2
    full_pass = strict_pass and wf_guard_pass
    return {
        'strict_pass': strict_pass,
        'wf_guard_pass': bool(wf_guard_pass),
        'full_pass': bool(full_pass),
        'strict_loss': float(strict_loss),
        'wf_loss': float(wf_loss),
        'wf_missing': int(wf_missing),
        'robust_loss': float(strict_loss + wf_loss + 1000.0 * wf_missing),
    }

outcomes = pd.DataFrame([outcome_fields(r) for _, r in screen_base.iterrows()])
screen_base = pd.concat([screen_base.reset_index(drop=True), outcomes], axis=1)
print(f'strict paths={len(screen_base)}, strict pass={screen_base.strict_pass.sum()}, full pass={screen_base.full_pass.sum()}')
display(screen_base.sort_values(['strict_loss', 'relative_return'], ascending=[True, False]).head(10)[['family','path','relative_return','max_dd','strict_pass','robust_loss','wf_relative_return_wf_2020','wf_max_dd_wf_2020','wf_relative_return_wf_2023_2024','wf_max_dd_wf_2023_2024']])


supplemental_wf_rows=6
strict paths=37, strict pass=2, full pass=0


,family,path,relative_return,max_dd,strict_pass,robust_loss,wf_relative_return_wf_2020,wf_max_dd_wf_2020,wf_relative_return_wf_2023_2024,wf_max_dd_wf_2023_2024
35,predictive_drawdown_risk_gate_v1,runtime/model_selection/card/predictive_drawdo...,35.443239,14.621835,True,293.436680,-42.486967,18.533823,39.485655,16.518242
11,risk_budget_residual_drawdown_guard_v1,runtime/execution/card/risk_budget_residual_dr...,30.457960,14.642461,True,517.483446,-54.194588,18.102212,38.310638,17.229977
7,risk_budget_crash_risk_factor_v1,runtime/execution/card/risk_budget_crash_risk_...,44.599109,15.484927,False,3000.940616,NaN,NaN,NaN,NaN
8,risk_budget_downside_vol_target_v1,runtime/execution/card/risk_budget_downside_vo...,31.059765,16.277862,False,3006.531723,NaN,NaN,NaN,NaN
9,risk_budget_downside_vol_target_v1,runtime/execution/card/risk_budget_downside_vo...,31.333073,16.277862,False,3006.531723,NaN,NaN,NaN,NaN
29,risk_budget_usage_v1,runtime/model/card/risk_budget_usage_v1/risk_b...,42.496862,21.103797,False,3149.025353,NaN,NaN,NaN,NaN
10,risk_budget_downside_vol_trendbreak_reentry_v1,runtime/execution/card/risk_budget_downside_vo...,13.130369,15.406526,False,3285.245489,NaN,NaN,NaN,NaN
27,risk_budget_usage_v1,runtime/model/card/risk_budget_usage_v1/risk_b...,2.767661,22.153538,False,3946.292714,NaN,NaN,NaN,NaN
0,recovery_reentry_2021_trendbreak_cap_v1,runtime/execution/card/recovery_reentry_2021_t...,207.197964,30.436657,False,1789.294593,25.557493,26.139320,46.096740,28.204933
2,recovery_reentry_conditional_vol_target_v1,runtime/execution/card/recovery_reentry_condit...,2171.722089,30.923275,False,1417.884530,55.796392,26.139320,68.890281,16.826953


In [4]:
def reward_score(df: pd.DataFrame, weights: dict[str, float]) -> pd.Series:
    return (
        df['z_log_return_sum']
        - weights['max_dd'] * df['z_max_dd']
        - weights['current_dd'] * df['z_current_dd_penalty']
        - weights['future_dd'] * df['z_future_dd_penalty']
        - weights['rebound_cost'] * df['z_rebound_opportunity_cost']
        - weights['bear_exposure'] * df['z_bear_exposure_penalty']
        - weights['turnover'] * df['z_turnover_per_day']
    )

weight_grid = []
for max_dd_w, current_dd_w, future_dd_w, rebound_w, bear_w, turnover_w in itertools.product(
    [0.0, 0.75, 1.5, 3.0],
    [0.0, 0.75, 1.5],
    [0.0, 0.75, 1.5],
    [0.0, 0.75, 1.5, 3.0],
    [0.0, 0.75, 1.5],
    [0.0, 0.5],
):
    # Skip nearly empty return-only duplicates except the explicit return-only baseline.
    weight_grid.append({
        'max_dd': max_dd_w,
        'current_dd': current_dd_w,
        'future_dd': future_dd_w,
        'rebound_cost': rebound_w,
        'bear_exposure': bear_w,
        'turnover': turnover_w,
    })

variant_rows = []
for i, weights in enumerate(weight_grid):
    scored = screen_base.copy()
    scored['reward_proxy_score'] = reward_score(scored, weights)
    selected = scored.sort_values(['reward_proxy_score', 'relative_return'], ascending=[False, False]).iloc[0]
    top5 = scored.sort_values(['reward_proxy_score', 'relative_return'], ascending=[False, False]).head(5)
    variant_rows.append({
        'variant_id': f'proxy_{i:04d}',
        **{f'w_{k}': v for k, v in weights.items()},
        'selected_family': selected['family'],
        'selected_path': selected['path'],
        'selected_score': float(selected['reward_proxy_score']),
        'selected_relative_return': float(selected['relative_return']),
        'selected_max_dd': float(selected['max_dd']),
        'selected_avg_pos': float(selected['avg_pos']),
        'selected_rebound_cost': float(selected['rebound_opportunity_cost']),
        'selected_turnover_per_day': float(selected['turnover_per_day']),
        'strict_pass': bool(selected['strict_pass']),
        'wf_guard_pass': bool(selected['wf_guard_pass']),
        'full_pass': bool(selected['full_pass']),
        'strict_loss': float(selected['strict_loss']),
        'wf_loss': float(selected['wf_loss']),
        'wf_missing': int(selected['wf_missing']),
        'robust_loss': float(selected['robust_loss']),
        'wf_2020_relative': float(selected.get('wf_relative_return_wf_2020', np.nan)),
        'wf_2020_max_dd': float(selected.get('wf_max_dd_wf_2020', np.nan)),
        'wf_2022_relative': float(selected.get('wf_relative_return_wf_2022', np.nan)),
        'wf_2022_max_dd': float(selected.get('wf_max_dd_wf_2022', np.nan)),
        'wf_2023_2024_relative': float(selected.get('wf_relative_return_wf_2023_2024', np.nan)),
        'wf_2023_2024_max_dd': float(selected.get('wf_max_dd_wf_2023_2024', np.nan)),
        'top5_families': '|'.join(top5['family'].astype(str).tolist()),
    })

variant_table = pd.DataFrame(variant_rows)
print(f'variants={len(variant_table)}, full_pass selections={variant_table.full_pass.sum()}, strict_pass selections={variant_table.strict_pass.sum()}')
display(variant_table.sort_values(['full_pass','robust_loss','strict_loss'], ascending=[False, True, True]).head(20))


variants=864, full_pass selections=0, strict_pass selections=338


,variant_id,w_max_dd,w_current_dd,w_future_dd,w_rebound_cost,w_bear_exposure,w_turnover,selected_family,selected_path,selected_score,selected_relative_return,selected_max_dd,selected_avg_pos,selected_rebound_cost,selected_turnover_per_day,strict_pass,wf_guard_pass,full_pass,strict_loss,wf_loss,wf_missing,robust_loss,wf_2020_relative,wf_2020_max_dd,wf_2022_relative,wf_2022_max_dd,wf_2023_2024_relative,wf_2023_2024_max_dd,top5_families
52,proxy_0052,0.00,0.00,1.50,0.00,1.50,0.0,predictive_drawdown_risk_gate_v1,runtime/model_selection/card/predictive_drawdo...,3.088700,35.443239,14.621835,0.161365,2.33978,0.022393,True,False,False,0.0,293.43668,0,293.43668,-42.486967,18.533823,76.657302,10.970665,39.485655,16.518242,predictive_drawdown_risk_gate_v1|risk_budget_c...
58,proxy_0058,0.00,0.00,1.50,0.75,1.50,0.0,predictive_drawdown_risk_gate_v1,runtime/model_selection/card/predictive_drawdo...,2.980871,35.443239,14.621835,0.161365,2.33978,0.022393,True,False,False,0.0,293.43668,0,293.43668,-42.486967,18.533823,76.657302,10.970665,39.485655,16.518242,predictive_drawdown_risk_gate_v1|risk_budget_c...
124,proxy_0124,0.00,0.75,1.50,0.00,1.50,0.0,predictive_drawdown_risk_gate_v1,runtime/model_selection/card/predictive_drawdo...,3.748566,35.443239,14.621835,0.161365,2.33978,0.022393,True,False,False,0.0,293.43668,0,293.43668,-42.486967,18.533823,76.657302,10.970665,39.485655,16.518242,predictive_drawdown_risk_gate_v1|risk_budget_c...
130,proxy_0130,0.00,0.75,1.50,0.75,1.50,0.0,predictive_drawdown_risk_gate_v1,runtime/model_selection/card/predictive_drawdo...,3.640737,35.443239,14.621835,0.161365,2.33978,0.022393,True,False,False,0.0,293.43668,0,293.43668,-42.486967,18.533823,76.657302,10.970665,39.485655,16.518242,predictive_drawdown_risk_gate_v1|risk_budget_c...
136,proxy_0136,0.00,0.75,1.50,1.50,1.50,0.0,predictive_drawdown_risk_gate_v1,runtime/model_selection/card/predictive_drawdo...,3.532909,35.443239,14.621835,0.161365,2.33978,0.022393,True,False,False,0.0,293.43668,0,293.43668,-42.486967,18.533823,76.657302,10.970665,39.485655,16.518242,predictive_drawdown_risk_gate_v1|risk_budget_c...
196,proxy_0196,0.00,1.50,1.50,0.00,1.50,0.0,predictive_drawdown_risk_gate_v1,runtime/model_selection/card/predictive_drawdo...,4.408432,35.443239,14.621835,0.161365,2.33978,0.022393,True,False,False,0.0,293.43668,0,293.43668,-42.486967,18.533823,76.657302,10.970665,39.485655,16.518242,predictive_drawdown_risk_gate_v1|risk_budget_c...
202,proxy_0202,0.00,1.50,1.50,0.75,1.50,0.0,predictive_drawdown_risk_gate_v1,runtime/model_selection/card/predictive_drawdo...,4.300603,35.443239,14.621835,0.161365,2.33978,0.022393,True,False,False,0.0,293.43668,0,293.43668,-42.486967,18.533823,76.657302,10.970665,39.485655,16.518242,predictive_drawdown_risk_gate_v1|risk_budget_c...
208,proxy_0208,0.00,1.50,1.50,1.50,1.50,0.0,predictive_drawdown_risk_gate_v1,runtime/model_selection/card/predictive_drawdo...,4.192775,35.443239,14.621835,0.161365,2.33978,0.022393,True,False,False,0.0,293.43668,0,293.43668,-42.486967,18.533823,76.657302,10.970665,39.485655,16.518242,predictive_drawdown_risk_gate_v1|risk_budget_c...
220,proxy_0220,0.75,0.00,0.00,0.00,1.50,0.0,predictive_drawdown_risk_gate_v1,runtime/model_selection/card/predictive_drawdo...,2.173499,35.443239,14.621835,0.161365,2.33978,0.022393,True,False,False,0.0,293.43668,0,293.43668,-42.486967,18.533823,76.657302,10.970665,39.485655,16.518242,predictive_drawdown_risk_gate_v1|risk_budget_r...
226,proxy_0226,0.75,0.00,0.00,0.75,1.50,0.0,predictive_drawdown_risk_gate_v1,runtime/model_selection/card/predictive_drawdo...,2.065671,35.443239,14.621835,0.161365,2.33978,0.022393,True,False,False,0.0,293.43668,0,293.43668,-42.486967,18.533823,76.657302,10.970665,39.485655,16.518242,predictive_drawdown_risk_gate_v1|risk_budget_r...


## Term Diagnostics

This section checks which terms actually change selection behavior. The key risk is a scalar reward that simply moves between two bad extremes:

- high-return recovery paths with 30%+ strict MaxDD;
- low-drawdown selector paths that underparticipate in `wf_2020` rebounds.


In [5]:
selected_counts = variant_table.groupby('selected_family').size().rename('selected_count').reset_index().sort_values('selected_count', ascending=False)
selected_quality = (
    variant_table.groupby('selected_family')
    .agg(
        selected_count=('variant_id', 'count'),
        strict_pass_rate=('strict_pass', 'mean'),
        full_pass_rate=('full_pass', 'mean'),
        min_robust_loss=('robust_loss', 'min'),
        median_robust_loss=('robust_loss', 'median'),
        median_strict_dd=('selected_max_dd', 'median'),
        median_relative=('selected_relative_return', 'median'),
        median_wf_2020_relative=('wf_2020_relative', 'median'),
        median_wf_2023_2024_dd=('wf_2023_2024_max_dd', 'median'),
    )
    .reset_index()
    .sort_values(['full_pass_rate','min_robust_loss'], ascending=[False, True])
)

print('Selected-family distribution')
display(selected_quality.head(30))

term_effects = []
for term in ['w_max_dd','w_current_dd','w_future_dd','w_rebound_cost','w_bear_exposure','w_turnover']:
    grouped = variant_table.groupby(term).agg(
        median_robust_loss=('robust_loss','median'),
        best_robust_loss=('robust_loss','min'),
        strict_pass_rate=('strict_pass','mean'),
        median_selected_dd=('selected_max_dd','median'),
        median_selected_relative=('selected_relative_return','median'),
        median_wf2020_relative=('wf_2020_relative','median'),
    ).reset_index()
    grouped.insert(0, 'term', term)
    grouped = grouped.rename(columns={term: 'weight'})
    term_effects.append(grouped)
term_effect_table = pd.concat(term_effects, ignore_index=True)
display(term_effect_table)


Selected-family distribution


,selected_family,selected_count,strict_pass_rate,full_pass_rate,min_robust_loss,median_robust_loss,median_strict_dd,median_relative,median_wf_2020_relative,median_wf_2023_2024_dd
3,predictive_drawdown_risk_gate_v1,338,1.0,0.0,293.436680,293.436680,14.621835,35.443239,-42.486967,16.518242
4,recovery_reentry_conditional_vol_target_v1,149,0.0,0.0,1417.884530,1417.884530,30.923275,2171.722089,55.796392,16.826953
5,recovery_reentry_cppi_floor_v1,5,0.0,0.0,1883.764837,1883.764837,32.120180,1594.473476,55.796392,30.183184
9,soxl_soxx_efficiency_chop_guard_v1,5,0.0,0.0,2049.667473,2049.667473,31.957679,595.207043,25.816839,30.512835
6,risk_budget_crash_risk_factor_v1,8,0.0,0.0,3000.940616,3000.940616,15.484927,44.599109,NaN,NaN
8,risk_budget_usage_v1,66,0.0,0.0,3149.025353,3149.025353,21.103797,42.496862,NaN,NaN
7,risk_budget_downside_vol_trendbreak_reentry_v1,39,0.0,0.0,3285.245489,3285.245489,15.406526,13.130369,NaN,NaN
0,balanced_rule_residual_mode_v1,7,0.0,0.0,5930.637160,5930.637160,48.137196,511.848031,22.204088,31.924701
2,balanced_rule_residual_v19_small,175,0.0,0.0,7267.284045,7267.284045,43.963716,-0.194314,NaN,NaN
1,balanced_rule_residual_v19_lowcap,2,0.0,0.0,8162.115719,8162.115719,50.923933,343.916374,NaN,NaN


,term,weight,median_robust_loss,best_robust_loss,strict_pass_rate,median_selected_dd,median_selected_relative,median_wf2020_relative
0,w_max_dd,0.00,7267.284045,293.43668,0.037037,43.963716,1252.131881,55.796392
1,w_max_dd,0.75,3149.025353,293.43668,0.189815,30.923275,35.443239,55.796392
2,w_max_dd,1.50,1417.884530,293.43668,0.416667,21.103797,35.443239,-42.486967
3,w_max_dd,3.00,293.436680,293.43668,0.921296,14.621835,35.443239,-42.486967
4,w_current_dd,0.00,1417.884530,293.43668,0.350694,30.923275,35.443239,-42.486967
5,w_current_dd,0.75,1417.884530,293.43668,0.395833,26.013536,35.443239,-42.486967
6,w_current_dd,1.50,1417.884530,293.43668,0.427083,21.103797,35.443239,-42.486967
7,w_future_dd,0.00,1417.884530,293.43668,0.305556,30.923275,35.443239,-42.486967
8,w_future_dd,0.75,1417.884530,293.43668,0.378472,30.923275,35.443239,-42.486967
9,w_future_dd,1.50,1417.884530,293.43668,0.489583,15.406526,35.443239,-42.486967


In [6]:
best_variant = variant_table.sort_values(['full_pass','robust_loss','strict_loss'], ascending=[False, True, True]).iloc[0]
return_only = variant_table[(variant_table[['w_max_dd','w_current_dd','w_future_dd','w_rebound_cost','w_bear_exposure','w_turnover']] == 0).all(axis=1)].iloc[0]
strict_best_path = screen_base.sort_values(['strict_loss','relative_return'], ascending=[True, False]).iloc[0]
robust_best_path = screen_base.sort_values(['robust_loss','strict_loss'], ascending=[True, True]).iloc[0]

summary = {
    'experiment': 'drawdown_aware_reward_proxy_v1',
    'decision': 'diagnostic_no_promotion',
    'path_pool_size': int(len(path_table)),
    'strict_path_count': int(len(screen_base)),
    'walkforward_path_count': int(len(wf_paths)),
    'reward_proxy_variants': int(len(variant_table)),
    'variant_full_pass_count': int(variant_table['full_pass'].sum()),
    'variant_strict_pass_count': int(variant_table['strict_pass'].sum()),
    'return_only_selection': {
        'family': str(return_only['selected_family']),
        'relative_return': float(return_only['selected_relative_return']),
        'max_dd': float(return_only['selected_max_dd']),
        'robust_loss': float(return_only['robust_loss']),
    },
    'best_proxy_selection': {
        'variant_id': str(best_variant['variant_id']),
        'family': str(best_variant['selected_family']),
        'path': str(best_variant['selected_path']),
        'relative_return': float(best_variant['selected_relative_return']),
        'max_dd': float(best_variant['selected_max_dd']),
        'strict_pass': bool(best_variant['strict_pass']),
        'wf_guard_pass': bool(best_variant['wf_guard_pass']),
        'full_pass': bool(best_variant['full_pass']),
        'robust_loss': float(best_variant['robust_loss']),
        'weights': {
            'max_dd': float(best_variant['w_max_dd']),
            'current_dd': float(best_variant['w_current_dd']),
            'future_dd': float(best_variant['w_future_dd']),
            'rebound_cost': float(best_variant['w_rebound_cost']),
            'bear_exposure': float(best_variant['w_bear_exposure']),
            'turnover': float(best_variant['w_turnover']),
        },
        'wf_2020_relative': float(best_variant['wf_2020_relative']),
        'wf_2020_max_dd': float(best_variant['wf_2020_max_dd']),
        'wf_2023_2024_relative': float(best_variant['wf_2023_2024_relative']),
        'wf_2023_2024_max_dd': float(best_variant['wf_2023_2024_max_dd']),
    },
    'best_available_path_by_robust_loss': {
        'family': str(robust_best_path['family']),
        'path': str(robust_best_path['path']),
        'relative_return': float(robust_best_path['relative_return']),
        'max_dd': float(robust_best_path['max_dd']),
        'robust_loss': float(robust_best_path['robust_loss']),
        'strict_pass': bool(robust_best_path['strict_pass']),
        'full_pass': bool(robust_best_path['full_pass']),
    },
    'strict_best_path': {
        'family': str(strict_best_path['family']),
        'path': str(strict_best_path['path']),
        'relative_return': float(strict_best_path['relative_return']),
        'max_dd': float(strict_best_path['max_dd']),
        'robust_loss': float(strict_best_path['robust_loss']),
    },
    'conclusion': [
        'No scalar reward proxy in the screened grid selects a full-pass candidate from the current path pool.',
        'Drawdown-heavy terms push selection toward lower-risk paths but preserve wf_2020 rebound underparticipation.',
        'Rebound/opportunity terms push selection toward high-return paths but strict MaxDD remains far above 15%.',
        'The next training branch should be regime-conditioned: drawdown penalty in broken-anchor/downtrend states plus explicit rebound opportunity in high-volatility positive-momentum states.',
    ],
    'next_training_branch': {
        'role': 'diagnose_or_ablate',
        'recommended_design': 'state-conditioned reward or auxiliary critic rather than another scalar reward-weight sweep',
        'terms': [
            'negative account drawdown / expected future drawdown term active in downtrend or broken-anchor states',
            'opportunity-cost term active only in high-volatility positive-momentum rebound states',
            'walk-forward weighted checkpoint selection loss',
            'turnover penalty retained as a small regularizer',
        ],
        'avoid': [
            'global scalar drawdown penalty alone',
            'global rebound bonus without 2022 defense gate',
            'broad PPO retrain without walk-forward checkpoint selection',
        ],
    },
    'artifacts': {
        'notebook': str((CARD_DIR / 'drawdown_aware_reward_proxy_v1.ipynb').relative_to(ROOT)),
        'path_reward_terms': str((CARD_DIR / 'path_reward_terms.csv').relative_to(ROOT)),
        'reward_proxy_screen': str((CARD_DIR / 'reward_proxy_screen.csv').relative_to(ROOT)),
        'reward_proxy_variant_screen': str((CARD_DIR / 'reward_proxy_variant_screen.csv').relative_to(ROOT)),
        'reward_proxy_family_summary': str((CARD_DIR / 'reward_proxy_family_summary.csv').relative_to(ROOT)),
        'reward_proxy_term_effects': str((CARD_DIR / 'reward_proxy_term_effects.csv').relative_to(ROOT)),
        'selection': str((CARD_DIR / 'selection.json').relative_to(ROOT)),
    },
}

path_table.to_csv(CARD_DIR / 'path_reward_terms.csv', index=False)
screen_base.to_csv(CARD_DIR / 'reward_proxy_screen.csv', index=False)
variant_table.to_csv(CARD_DIR / 'reward_proxy_variant_screen.csv', index=False)
selected_quality.to_csv(CARD_DIR / 'reward_proxy_family_summary.csv', index=False)
term_effect_table.to_csv(CARD_DIR / 'reward_proxy_term_effects.csv', index=False)
with (CARD_DIR / 'selection.json').open('w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))


{
  "experiment": "drawdown_aware_reward_proxy_v1",
  "decision": "diagnostic_no_promotion",
  "path_pool_size": 88,
  "strict_path_count": 37,
  "walkforward_path_count": 51,
  "reward_proxy_variants": 864,
  "variant_full_pass_count": 0,
  "variant_strict_pass_count": 338,
  "return_only_selection": {
    "family": "recovery_reentry_conditional_vol_target_v1",
    "relative_return": 2171.722089100399,
    "max_dd": 30.923275332666435,
    "robust_loss": 1417.8845304558718
  },
  "best_proxy_selection": {
    "variant_id": "proxy_0052",
    "family": "predictive_drawdown_risk_gate_v1",
    "path": "runtime/model_selection/card/predictive_drawdown_risk_gate_v1/predictive_drawdown_risk_gate_v1_selected_soxl_daily.csv",
    "relative_return": 35.443239071048936,
    "max_dd": 14.621835439928654,
    "strict_pass": true,
    "wf_guard_pass": false,
    "full_pass": false,
    "robust_loss": 293.43667979127247,
    "weights": {
      "max_dd": 0.0,
      "current_dd": 0.0,
      "future_dd

## Notebook Decision

This notebook should be treated as **diagnostic no-promotion** unless `variant_full_pass_count > 0`.

If no scalar proxy finds a full-pass candidate, the next meaningful loop is a state-conditioned reward design or auxiliary-risk architecture, not another scalar penalty sweep.
